## 5.0 Introduction

In Lecture 5 you saw the full unsupervised pipeline applied to the Iris dataset: standardize, run k-means with Scikit-learn, reduce to two dimensions with PCA, and visualize. In this homework you repeat that pipeline on two new datasets.

**Section 5.1–5.5** apply the pipeline to the Palmer Penguins dataset. The workflow mirrors the lecture exactly — you will standardize, cluster, cross-tabulate, reduce with PCA, and visualize — so you can focus on interpreting the results rather than figuring out the mechanics.

**Section 5.6** applies k-means to the College dataset, a survey of 777 U.S. universities. Unlike Iris and Penguins, there is no ground-truth label to evaluate against. You will use the elbow plot to choose $k$, then explore which universities k-means grouped with the University of Washington.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 5.1 Loading and Preparing the Penguins Dataset

We'll apply the full k-means pipeline to the Palmer Penguins dataset. You cleaned this dataset in Homework 3, so the loading steps below should look familiar.

In [ ]:
url = ?? # Fill this in. 
df = ?? # Finish this. 
 
print('Shape before cleaning:', df.shape)
df_clean = ?? # Finish this by dropping rows with missing values.
print('Shape after dropping missing rows:', df_clean.shape)
df_clean.head()

In [ ]:
feature_names = # Finish this. 

X = # Finish this and convert from a DataFrame to a NumPy array.
y = # Finish this by making: 0 = Adelie, 1 = Chinstrap, 2 = Gentoo
species_names = list(pd.Categorical(df_clean['species']).categories)

print('X shape:', X.shape)
print('Species encoding:', {i: name for i, name in enumerate(species_names)})

## 5.2 Standardize

In [ ]:
# Import the StandardScaler here. 

scaler = # Initialize the StandardScaler here.
X_std = # Fit the scaler to X and transform X here.

n = # Finish this. 
print(f"{'Feature':<22} {'Norm':>10}")
print('-' * 35)
for j, name in enumerate(feature_names):
    print(f'{name:<22} {np.linalg.norm(X_std[:, j]):>10.4f}')

**Exercise 1.** The above table helps you verify that the standardization process worked; it checks to confirm that the magnitudes of the feature vectors are all $\sqrt{n} = $ \_\_\_\_\_. 

## 5.3 k-Means with Scikit-learn

Run k-means with $k = 3$ clusters — one for each species. Use `n_init=10` independent runs and `random_state=0` for reproducibility.

In [ ]:
# Import KMeans from sklearn.cluster here. 

kmeans = # Initialize here. 
labels = kmeans.fit_predict(X_std)

print('Cluster label counts:', np.bincount(labels))

In [ ]:
centers = kmeans.cluster_centers_
print('Cluster centroids (standardized):')
print(np.round(centers, 4))

In [ ]:
# Run this cell without modification.
print('Cross-tabulation: rows = true species, columns = k-means cluster')
print(f"{'':>15}  cluster 0  cluster 1  cluster 2      total")
table = []
for sp, name in enumerate(species_names):
    row = [int(np.sum((y == sp) & (labels == j))) for j in range(3)]
    table.append(row)
    print(f'{name:>15}: {row[0]:>9}  {row[1]:>9}  {row[2]:>9}  {sum(row):>9}')
col_totals = [sum(table[sp][j] for sp in range(3)) for j in range(3)]
print(f"{'total':>15}: {col_totals[0]:>9}  {col_totals[1]:>9}  {col_totals[2]:>9}  {sum(col_totals):>9}")

**Exercise 2.** The \_\_\_\_\_\_ species is recovered perfectly. \_\_\_\_\_\_ is the hardest to separate. Evaluating using a table like this is unrealistic for real-world k-means because it's an \_\_\_\_\_\_ learning algorithm. 

## 5.4 PCA

**Exercise 3.** After cleaning the data and keeping only the numerical values, each individual penguin is represented by a \_\_\_\_ vector that lives in \_\_\_ dimensional space. This is too many dimensions to visualize directly, so we use PCA to project down to \_\_\_ dimensions while preserving as much variance as possible.

Finish the code cell below to make this happen. 

In [ ]:
# Import PCA from sklearn.decomposition here.

pca = # Initialize PCA here.
V = # Fit PCA to the standardized data and get the principal components here. 

print('V shape:', V.shape)
print('First 3 rows of V:')
print(np.round(V[:3, :], 4))

**Exercise 4.** `V[2, 1]` = \_\_\_\_\_. 

In [ ]:
# Run this cell without modification.
evr = pca.explained_variance_ratio_
print(f'PC1 explains {evr[0]*100:.1f}% of total variance')
print(f'PC2 explains {evr[1]*100:.1f}% of total variance')
print(f'Together:    {sum(evr)*100:.1f}%')

**Exercise 5.** How much variance (in percent) do PC1 and PC2 together explain? Round to 1 decimal place. 

**Question.** The Iris dataset had 4 features and two components explained about 96% of the variance. The Penguins dataset also has 4 features. Do the first two components explain a similar fraction? What might this tell you about the intrinsic structure of the two datasets and the informational usefulness of the features?

**Exercise 6.** Because the PCA uses fewer dimensions than the original data there is informational loss. After the PCA is applied, which loses more information: **Iris** or **Penguins**? 

## 5.5 Visualizing k-means Results via PCA

Now we have k-means labels and PCA coordinates. Make two side-by-side plots — the same layout as Section 5.6 of the lecture:

1. PCA coordinates colored by **k-means cluster label** — colors matched to their closest species by majority vote
2. PCA coordinates colored by **true species label** — same color per species as plot 1.

In [ ]:
# Run this cell without modification.

# Align k-means cluster colors to true species by majority vote
color_map = {}
for c in range(3):
    counts = [int(np.sum((labels == c) & (y == k))) for k in range(3)]
    color_map[c] = int(np.argmax(counts))

colors = ['steelblue', 'tomato', 'seagreen']   # indexed by species code
cluster_colors = [colors[color_map[c]] for c in range(3)]

# Project k-means centroids into PCA space
centers_pca = pca.transform(centers)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: k-means clusters
ax = axes[0]
for c in range(3):
    mask = (labels == c)
    matched_name = species_names[color_map[c]]
    ax.scatter(V[mask, 0], V[mask, 1], color=cluster_colors[c],
               label=f'Cluster {c} ({matched_name})', alpha=0.7,
               edgecolors='white', linewidth=0.5)
for c in range(3):
    ax.scatter(centers_pca[c, 0], centers_pca[c, 1],
               color=cluster_colors[c], marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('k-means clusters (PCA view)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# Plot 2: true species
ax = axes[1]
for k, name in enumerate(species_names):
    mask = (y == k)
    centroid_pca = pca.transform(np.mean(X_std[mask, :], axis=0).reshape(1, -1))
    ax.scatter(V[mask, 0], V[mask, 1], color=colors[k],
               label=name, alpha=0.7, edgecolors='white', linewidth=0.5)
    ax.scatter(centroid_pca[0, 0], centroid_pca[0, 1],
               color=colors[k], marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('True species (PCA view)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Penguins: k-means Clusters vs. True Species (PCA view)', fontsize=13)
plt.tight_layout()
plt.show()

**Question.** Compare the two plots. The color alignment makes mismatches easy to spot — a point that is steelblue in the left plot but tomato in the right is a misclassified penguin. Where do the cluster assignments and the true species labels disagree? Cross-reference with the cross-tabulation from Section 5.3.

## 5.6 k-means on the College Dataset

The [College dataset](https://www.kaggle.com/datasets/flyingwombat/us-news-and-world-reports-college-data) contains data on 777 U.S. colleges and universities from a U.S. News & World Report survey. Each row is one institution described by 17 numerical features — tuition, graduation rate, student-to-faculty ratio, and so on — plus two non-numerical columns: `Name` and `Private`. We'll drop those before clustering.

Unlike Iris and Penguins, there is no ground-truth label to check against; people debate this all the time! This is the more realistic unsupervised setting: we are clustering because we don't know the groups in advance. That means we also have to choose $k$ ourselves.

In [ ]:
url = # Finish this; the college.csv dataset is in the course repository. 
df_college = # Finish this. There are no missing values in this dataset, so we don't need to drop any rows.

print('Shape:', df_college.shape)
df_college.head()

In [ ]:
# Run this cell without modification to drop the non-numeric columns and convert to a NumPy array.
X_college = df_college.drop(columns=['Name', 'Private']).to_numpy()
print('X_college shape:', X_college.shape)

**Exercise 7.** There are \_\_\_ rows and \_\_\_ columns in `X_college`.

In [ ]:
# Import the StandardScaler here.
scaler_college = # Initialize the StandardScaler here.
X_college_std = # Fit the scaler to X_college and transform X_college here.

### Choosing $k$: the Elbow Plot

With Iris and Penguins we set the $k = 3$ because we knew the number of species. Here we don't know the right number of clusters. Plot $J^{\text{clust}}$ as a function of $k$ and look for an **elbow** — the point where adding another cluster stops producing a meaningful reduction in the objective. 

Elbow plots to help choose the value of a **hyperparameter** are commonplace in modern machine learning; you see them come up in all kinds of modeling problems. 

In [ ]:
# Run this cell without modification. 
from sklearn.cluster import KMeans

J_values = []
k_values = range(1, 13)

for k_val in k_values:
    km = KMeans(n_clusters=k_val, n_init=10, random_state=0)
    km.fit(X_college_std)
    J_values.append(km.inertia_ / X_college_std.shape[0])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, J_values, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('$k$ (number of clusters)')
ax.set_ylabel('$J^{\text{clust}}$ (normalized)')
ax.set_title('Elbow plot: College dataset')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Exercise 8.** Elbow plots require a lot of compute power; this isn't a big deal these days so elbow plots are quite common. How many total k-means models are created and trained in the above code cell? 

Hint: It may be helpful to look at what `for val in range(1,13): print(val)` does in a separate code cell. What does `n_init` do?

**Exercise 9.** Based on the elbow plot, what values of $k$ where the plot *starts to elbow* and therefore see less of a reduction in $J^{clust}$ for increases in $k$ happen around $k =$ \_\_\_ or $k =$ \_\_\_. 

In [ ]:
# Set your chosen value of hyperparameter k here. 
k = ??

Now run k-means with your chosen `k` and reduce to 2 dimensions with PCA.

In [ ]:
# Run this cell without modification; notice that n_clusters is set to your chosen value of k from ealier.
from sklearn.decomposition import PCA

kmeans_college = KMeans(n_clusters=k, n_init=10, random_state=0)
labels_college = kmeans_college.fit_predict(X_college_std)

pca_college = PCA(n_components=2)
V_college = pca_college.fit_transform(X_college_std)
centers_college_pca = pca_college.transform(kmeans_college.cluster_centers_)

evr = pca_college.explained_variance_ratio_
print(f'PC1 explains {evr[0]*100:.1f}%')
print(f'PC2 explains {evr[1]*100:.1f}%')
print(f'Together:    {sum(evr)*100:.1f}%')

In [ ]:
colors_k = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(8, 6))
for c in range(k):
    mask = (labels_college == c)
    ax.scatter(V_college[mask, 0], V_college[mask, 1],
               color=colors_k[c % 10], alpha=0.6,
               edgecolors='white', linewidth=0.3, s=40, label=f'Cluster {c}')
    ax.scatter(centers_college_pca[c, 0], centers_college_pca[c, 1],
               color=colors_k[c % 10], marker='*', s=250,
               edgecolors='black', linewidth=0.8, zorder=4)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title(f'College dataset: k-Means ($k={k}$, PCA view)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Question.** PC1 and PC2 explain a smaller fraction of the total variance for the College dataset than they did for Iris or Penguins. What does this tell you about the dimensionality of the College data — and what are the limits of using a 2D PCA plot to evaluate cluster quality here?

### Which universities did k-means group with the University of Washington?

In [ ]:
# Run this cell without modification to find the cluster of University of Washington and print all colleges in the same cluster.
# This code makes use of the .index and .iloc methods briefly mentioned in Lecture 3!
UW_index = df_college[df_college['Name'] == 'University of Washington'].index[0]
UW_cluster = labels_college[UW_index]
print(f'University of Washington is in cluster {UW_cluster}')
print()
print('All colleges in the same cluster:')
for i in range(X_college.shape[0]):
    if labels_college[i] == UW_cluster:
        print(f'  {df_college.iloc[i]["Name"]}')

### How much does where you attend college matter?

Conventional and cultural wisdom tell us that students should be very concerned with how "elite" their college of choice is. There is certainly some truth to this: name-brand institutions can provide better education, better connections to opportunities, and open doors that would otherwise be closed at "lesser" institutions. That said, the values of $k$ suggested by the elbow plot should make us seriously question how much one should worry about going to a "top-ranked" college versus a "good" one, at least when comparing institutions based on the numeric feature variables in the dataset. 

Just for fun, let's look at the the groupings we get if we take $k = 10$. Keep in mind that group number has nothing to do with any notion of "rank"; it's an arbitrary value assigned at runtime to identify separate groups, not compare them. When I run the **College** example on my machine UW belongs to cluster 1; that may or may not happen when you run the example in your environment. The group number doesn't matter; it's group membership that is interesting here. 

**Exercise 10.** The first college in this group (by alphabetical order) is \_\_\_\_\_\_\_\_\_\_ \_\_\_\_\_\_\_\_\_\_\_.

Note: Final k-means groupings are environment and version sensitive! Answers will vary: I've put the most common answers from the local and cloud environments into Canvas. If you want to discuss why this is the case (which is kind of surprising given we're setting the `random_state` parameter) I am happy to discuss this with you!

## 5.7 Free Response Exam Questions

**Exercise.** What is the scikit-learn library, what do we use it for, and what code do we use to access it?

**Exercise.** Who would find the college grouping results interesting? What kinds of people could make use of the results and how would they do so?

**Exercise.** Explain how ```random_state``` variable works in the context of ```cluster.KMeans()```. Why do we this?

**Exercise.** Why do we need to standardize real data before we run the k-Means algorithm?

**Exercise.** In Homework 4 we initialized centroids close together, which caused a badly skewed first assignment that took several iterations to correct. How does Scikit-learn's `n_init` parameter guard against this kind of poor initialization? Why do we keep the run with the lowest $J^{\text{clust}}$ rather than, say, the run that converges fastest?

**Exercise.** PCA and k-means are both unsupervised — neither uses species labels. Yet when we plot the PCA projection of the Penguins data colored by species, the clusters are clearly visible. What does this tell you about the relationship between the geometry of the data and the species labels?

**Exercise.** The College dataset has no ground-truth labels to evaluate the clustering against. In the Penguins section we used the species labels to check how well k-means did. Explain why having ground-truth labels made that evaluation possible, and what we lose when they aren't available. What kinds of evidence could you use instead to judge whether a clustering is good?

**Exercise.** How did the informational loss from PCA for the **College** dataset compare to the **Iris** and **Penguins** PCA? Whis is this unsurprising? Hint: how many original feature vectors did the PCA project (compress) in these datasets?